In [0]:
%sql
WITH gmb AS (
  SELECT name
    , address
    , split_part(address, ',',2) as city
    , split_part(split_part(address, ',', -1), ' ',2) as state_id
    , split_part(split_part(address, ',', -1), ' ',3) as zip_code
    , category
    , lat
    , lon
    , main_image
    , open_hours
    , open_hours_updated
    , open_website
    , phone_number
    , place_id
    , price_range
    , rating
    , reviews
    , reviews_count
    , services_provided
    , url
  FROM polar_dais_hackathon_2025.bright_initiative.google_maps_businesses
  WHERE category RLIKE 'Health|Wellness|Psychotherapist|Speech pathologist|Physical therapy clinic|Sports medicine clinic|Diagnostic center|Eye care center|Ophthalmology clinic|Medical clinic|Mental health clinic|Crisis center|Addiction treatment center|Massage spa|Massage therapist|Chiropractor|Acupuncturist|Reiki therapist|Holistic medicine practitioner|Nutritionist|Yoga studio|Meditation center|Pilates studio|Indoor cycling|Personal trainer|Fitness center|Physical fitness program|Psychiatrist|Psychologist|Counselor|Counseling service|Community center|Support group|Recreation center|Child psychologist'
)
SELECT * FROM gmb
WHERE city ILIKE '%San Francisco%'
AND reviews_count > 0

In [0]:
import pandas as pd
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Define a simple UDF
@udf(StringType())
def get_health_service(city: str) -> str:


In [0]:
import pandas as pd
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_databricks import ChatDatabricks
from databricks.sdk import WorkspaceClient
import os

# configure workspace tokens
w = WorkspaceClient()
os.environ["DATABRICKS_HOST"] = w.config.host
os.environ["DATABRICKS_TOKEN"] = w.tokens.create(comment="for model serving", lifetime_seconds=1200).token_value

llm = ChatDatabricks(endpoint="databricks-llama-4-maverick")

def format_context(df: pd.DataFrame) -> str:
    """
    Converts the DataFrame into a JSON string to ensure all data is passed
    to the model without truncation. JSON is also a great format for structured data
    like you have in 'description_by_sections'.
    """
    return df.to_json(orient='records', indent=2)

def find_wellnesscenters(city: str) -> pd.DataFrame:
  """
  Returns a DataFrame containing the top 10 wellness centers in the given city.
  """
  query = f"""
    WITH gmb AS (
    SELECT name
        , address
        , split_part(address, ',',2) as city
        , split_part(split_part(address, ',', -1), ' ',2) as state_id
        , split_part(split_part(address, ',', -1), ' ',3) as zip_code
        , category
        , lat
        , lon
        , main_image
        , open_hours
        , open_hours_updated
        , open_website
        , phone_number
        , place_id
        , price_range
        , rating
        , reviews
        , reviews_count
        , services_provided
        , url
    FROM polar_dais_hackathon_2025.bright_initiative.google_maps_businesses
    WHERE category RLIKE 'Health|Wellness|Psychotherapist|Speech pathologist|Physical therapy clinic|Sports medicine clinic|Diagnostic center|Eye care center|Ophthalmology clinic|Medical clinic|Mental health clinic|Crisis center|Addiction treatment center|Massage spa|Massage therapist|Chiropractor|Acupuncturist|Reiki therapist|Holistic medicine practitioner|Nutritionist|Yoga studio|Meditation center|Pilates studio|Indoor cycling|Personal trainer|Fitness center|Physical fitness program|Psychiatrist|Psychologist|Counselor|Counseling service|Community center|Support group|Recreation center|Child psychologist'
    )
    SELECT * 
    FROM gmb
    WHERE city ILIKE '%{city}%'
        AND reviews_count > 0
    LIMIT 10
  """
  return format_context(spark.sql(query).toPandas())

# Define the prompt template for the LLM
PromptTemplate.from_template(
  """
    You are Bridgie — a warm, supportive wellness assistant. Your goal is to help users discover **local, low-barrier wellness resources** and community services that match their needs, preferences, and constraints.

    You will have access to structured and unstructured data (community centers, wellness activities, support groups, reviews, descriptions, etc.).

    When responding, you should:
    - Use a **conversational, empathetic, friendly tone** — like a helpful wellness guide.
    - Acknowledge the user's feelings and goals with empathy.
    - Ask clarifying questions when needed (location, needs, cost sensitivity, mobility considerations).
    - Recommend **specific activities or resources**, ideally free or low-cost.
    - Summarize key details (name of the resource, location, cost, brief description).
    - Help the user build a simple **personalized wellness plan** if appropriate.

    The user may have mental health needs, limited budget, or mobility constraints — always be supportive and non-judgmental.

    You will be provided with JSON data containing potential wellness resources:
    {context}

    Review the data and suggest the most relevant options, with short explanations the user can easily understand.

    If no resources are found, offer encouragement and suggest general wellness tips (mindfulness, gentle movement, online resources).  
  """
)

llm = ChatDatabricks(endpoint="databricks-llama-4-maverick")

# This is our simple "agentic" chain
chain = (
    find_wellnesscenters
    | prompt_template
    | llm
    | StrOutputParser()
)


In [0]:
%pip install unitycatalog-client unitycatalog-ai unitycatalog

In [0]:
%restart_python

In [0]:
# from databricks.sdk import WorkspaceClient
# client = WorkspaceClient()
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
client = DatabricksFunctionClient()

def get_wellness_centers() -> str:
    """
    Returns a DataFrame containing the top 10 wellness centers in the given city.
    """
    query = f"""
      WITH gmb AS (
    SELECT name
      , address
      , split_part(address, ',',2) as city
      , split_part(split_part(address, ',', -1), ' ',2) as state_id
      , split_part(split_part(address, ',', -1), ' ',3) as zip_code
      , category
      , lat
      , lon
      , main_image
      , open_hours
      , open_hours_updated
      , open_website
      , phone_number
      , place_id
      , price_range
      , rating
      , reviews
      , reviews_count
      , services_provided
      , url
    FROM polar_dais_hackathon_2025.bright_initiative.google_maps_businesses
    WHERE category RLIKE 'Health|Wellness|Psychotherapist|Speech pathologist|Physical therapy clinic|Sports medicine clinic|Diagnostic center|Eye care center|Ophthalmology clinic|Medical clinic|Mental health clinic|Crisis center|Addiction treatment center|Massage spa|Massage therapist|Chiropractor|Acupuncturist|Reiki therapist|Holistic medicine practitioner|Nutritionist|Yoga studio|Meditation center|Pilates studio|Indoor cycling|Personal trainer|Fitness center|Physical fitness program|Psychiatrist|Psychologist|Counselor|Counseling service|Community center|Support group|Recreation center|Child psychologist'
  )
  SELECT * FROM gmb
  WHERE city ILIKE '%San Francisco%'
  AND reviews_count > 0
  LIMIT 10
  """
    return spark.sql(query).toJSON()  

function_info = client.create_python_function(
  func=get_wellness_centers,
  catalog="polar_dais",
  schema="polar",
  replace=True
)